# N7 — Reasoning

* Professor: Júlio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Alejandro Núñez Arroyo <a href="mailto:r299215@dac.unicamp.br">(r299215@dac.unicamp.br)</a>
* Material base: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a>

Este notebook trata do formato em que o raciocínio de um modelo chega à aplicação. Um modelo de raciocínio produz uma sequência de passos antes da resposta, e essa sequência fica em um canal separado que o provedor devolve sob um parâmetro próprio. As técnicas desta aula colocam os passos dentro da resposta, em um campo que o código da aplicação lê, exibe e verifica.

As seções vão de uma pergunta respondida sem técnica nenhuma até um arranjo que gera várias abordagens, pontua cada uma e resolve pela mais bem avaliada. O problema é o mesmo em todas, e muda a forma do raciocínio: um campo de texto, uma sequência de subperguntas, várias cadeias amostradas, um par de raciocínio e ação, uma ordem de subproblemas, ou um conjunto de candidatos avaliados.

## 0.1 Pré-requisitos

Este notebook pressupõe os seguintes conhecimentos:

- Python: funções, dicionários, listas, anotações de tipo e f-strings;
- declaração de esquema com Pydantic (`BaseModel` e `Field`) e saída estruturada com `with_structured_output`;
- papéis das mensagens de uma conversa: sistema, usuário e assistente;
- tool calling, o mecanismo pelo qual o modelo devolve na resposta o nome de uma função declarada e os argumentos dela, e a aplicação executa a função e devolve o retorno ao modelo;
- o par pedido e observação, gravado em `AIMessage.tool_calls` e em `ToolMessage`.

## 0.2 Objetivos da aula

Objetivos deste notebook:

1. Separar o raciocínio interno, gerado pelo modelo em um canal do provedor, do raciocínio escrito na resposta.
2. Recuperar o texto do raciocínio interno nos dois provedores e medir o que ele consome em tokens.
3. Induzir o formato de uma resposta por exemplos completos, com e sem os passos de raciocínio.
4. Separar raciocínio e conclusão em campos de um esquema Pydantic, com Chain-of-Thought.
5. Decompor a pergunta em subperguntas respondidas em sequência, com Self-Ask.
6. Amostrar várias cadeias independentes e decidir por votação, com Self-Consistency.
7. Intercalar raciocínio e chamada de ferramenta, com ReAct.
8. Resolver subproblemas em ordem de dependência, com Least-to-Most.
9. Gerar abordagens alternativas, pontuar cada uma e resolver pela escolhida, com Tree of Thoughts.
10. Comparar as sete técnicas pelo número de chamadas de modelo e pelo formato do raciocínio produzido.

## 0.3 Mapa do notebook

Conteúdo deste notebook:

1. Raciocínio interno e raciocínio na resposta.
2. Few-Shot Prompting.
3. Chain-of-Thought.
4. Self-Ask.
5. Self-Consistency.
6. ReAct.
7. Least-to-Most.
8. Tree of Thoughts.
9. Comparação das técnicas.
10. Exercícios de fixação.
11. Resumo da aula.
12. Referências.

## 0.4 Contexto usado nos exemplos

Os exemplos resolvem uma pergunta de planejamento do curso de agentes com LLMs. O aluno Renan ainda não começou os módulos A5 a A10, e a pergunta é se o conteúdo cabe no prazo que falta para a prova.

Todos os dados da pergunta estão no texto dela, e nenhuma seção consulta uma fonte externa para responder. Isso mantém a comparação entre as técnicas restrita ao formato do raciocínio. A seção 6 acrescenta uma ferramenta de cálculo, que não traz dado novo e serve para mostrar o raciocínio intercalado com a execução de uma função.

A pergunta tem uma resposta verificável e dois caminhos de cálculo que levam a conclusões diferentes:

| Caminho | Conta | Conclusão |
|---|---|---|
| Por minutos | 27 aulas × 50 min = 1350 min; 10 dias × 140 min = 1400 min; 1350 ≤ 1400 | termina a tempo |
| Por aulas inteiras | ⌊140 min ÷ 50 min⌋ = 2 aulas por dia; 2 × 10 = 20 aulas; 20 < 27 | não termina a tempo |

O segundo caminho é o correto, porque o enunciado diz que uma aula não pode ser dividida entre dois dias: os 40 minutos que sobram em cada dia não formam uma aula. A conclusão registrada é que Renan não termina o conteúdo nos 10 dias.

O primeiro caminho ignora a indivisibilidade e chega à conclusão oposta. Ele aparece nas execuções gravadas como passo intermediário descartado: as cadeias das seções 3 a 5 somam os 1350 minutos e passam à contagem de aulas inteiras no passo seguinte, e na seção 8 ele é uma das abordagens submetidas ao avaliador, que lhe dá a menor nota.

---

## 0.5 Preparação do ambiente

O notebook usa o modelo `gpt-oss:120b`, executado na infraestrutura do Ollama e acessado pela API em `https://ollama.com`. Nenhum modelo é baixado na máquina. A seção 0.6 registra também a configuração do Gemini, para o caso de a execução ser feita pela API do Google.

As bibliotecas são `langchain` (decorador `@tool`), `langchain-ollama` e `langchain-google-genai` (clientes dos dois provedores), `pydantic` (esquemas das saídas estruturadas) e `python-dotenv` (leitura da chave). A célula de imports carrega os dois clientes, então os dois pacotes precisam estar instalados; a constante `PROVEDOR` da seção 0.6 decide qual deles é instanciado.

Das seções 3 em diante, a execução depende de saída estruturada, o mecanismo pelo qual o modelo preenche os campos de um esquema declarado no pedido. A seção 6 depende de tool calling. O `gpt-oss:120b` e o `gemini-3.5-flash-lite` têm as duas capacidades. Um modelo sem saída estruturada devolve prosa onde o código espera um objeto, e a célula termina em erro de validação. As seções 1 e 2 rodam com chamadas de texto e não dependem de nenhuma das duas.

O acesso ao modelo hospedado exige uma chave de API. Os passos para obtê-la:

1. Criar uma conta em [ollama.com](https://ollama.com) e entrar nela.
2. Abrir [ollama.com/settings/keys](https://ollama.com/settings/keys) e gerar uma chave nova.
3. Copiar o valor exibido, que aparece uma única vez.
4. Guardar o valor na variável de ambiente `OLLAMA_API_KEY`.

Há três formas de cumprir o passo 4. Em uma máquina local, escreva a linha `OLLAMA_API_KEY=...` em um arquivo `.env` ao lado do notebook, que já está no `.gitignore`, e deixe o `load_dotenv()` da seção 0.6 carregá-la. A segunda forma é exportar a variável no ambiente antes de abrir o notebook. A terceira é a célula comentada abaixo, para o Google Colab. Não escreva a chave no código versionado.

In [1]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
# %pip install -U langchain langchain-ollama langchain-google-genai pydantic python-dotenv

A célula a seguir cobre o caso do Google Colab, onde não há arquivo `.env`. Descomente as três linhas, cole a chave entre as aspas e execute a célula. Torne a comentá-las antes de salvar o notebook, para que a chave não seja gravada no arquivo.

In [2]:
# # Descomente no Google Colab, onde não há arquivo .env.
# import os
# os.environ["OLLAMA_API_KEY"] = ""
# os.environ["GOOGLE_API_KEY"] = ""

Agora importamos os elementos que serão usados ao longo da aula.

In [3]:
import json
import math
import os

from collections import Counter

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.tools import tool
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

Glossário dos imports:

- `json`: serialização do retorno da ferramenta da seção 6;
- `math`: arredondamento para cima na conversão de aulas em dias;
- `os` e `load_dotenv`: leitura das chaves de API a partir do ambiente ou do arquivo `.env`;
- `Counter`: contagem dos votos entre as cadeias amostradas na seção 5;
- `BaseModel` e `Field`: declaração dos esquemas de saída estruturada e da descrição de cada campo;
- `tool`: decorador que converte uma função Python em ferramenta com nome, descrição e esquema;
- `AIMessage`, `HumanMessage`, `SystemMessage` e `ToolMessage`: a resposta do modelo, a pergunta, a instrução fixa e o resultado de uma ferramenta;
- `AnyMessage`: tipo comum às mensagens, usado nas anotações das funções auxiliares;
- `ChatOllama` e `ChatGoogleGenerativeAI`: clientes dos dois provedores configurados na seção 0.6.

As três funções auxiliares abaixo são usadas nas células de exemplo. A primeira imprime o raciocínio separado da conclusão, formato repetido em todas as técnicas. A segunda recupera o texto do raciocínio interno, que os dois provedores anexam à resposta em lugares diferentes. A terceira imprime uma conversa mensagem a mensagem.

In [4]:
def mostrar_raciocinio(raciocinio: str, conclusao: str) -> None:
    """Imprime o raciocínio e a conclusão em blocos separados."""
    print("RACIOCÍNIO")
    print(raciocinio)
    print("-" * 70)
    print("CONCLUSÃO")
    print(conclusao)


def raciocinio_interno(mensagem: AIMessage) -> str:
    """Devolve o texto do raciocínio interno, no campo em que cada provedor o entrega."""
    do_ollama = mensagem.additional_kwargs.get("reasoning_content")

    if do_ollama:
        return do_ollama

    return "\n".join(
        bloco["reasoning"] for bloco in mensagem.content_blocks if bloco["type"] == "reasoning"
    )


def mostrar_conversa(mensagens: list[AnyMessage]) -> None:
    """Imprime cada mensagem com o tipo, os pedidos de chamada e o conteúdo."""
    for indice, mensagem in enumerate(mensagens):
        print(f"[{indice}] {type(mensagem).__name__}")

        for chamada in getattr(mensagem, "tool_calls", []):
            print(f"    pedido: {chamada['name']}({chamada['args']})")

        if mensagem.text:
            rotulo = "resultado" if isinstance(mensagem, ToolMessage) else "texto"
            print(f"    {rotulo}: {mensagem.text[:400]}")

        print("-" * 70)

## 0.6 Escolhendo um provedor de LLM

O notebook instancia o modelo por uma função única, `criar_modelo`, e as seções seguintes chamam o objeto devolvido por ela. A constante `PROVEDOR` seleciona qual cliente é construído.

Os dois provedores diferem nos pontos da tabela abaixo, e `criar_modelo` concentra os que afetam a construção do cliente:

| Ponto | Ollama | Gemini |
|---|---|---|
| Classe | `ChatOllama` | `ChatGoogleGenerativeAI` |
| Conexão | `base_url` em `https://ollama.com` e cabeçalho `Authorization` entregue por `client_kwargs` | chave entregue em `api_key` |
| Limite de tokens da resposta | `num_predict` | `max_output_tokens` |
| Amostragem | `temperature`, de 0 a 1 | fixa; a família Gemini 3.x não aceita `temperature`, `top_p` nem `top_k` |
| Raciocínio interno | `reasoning`, ligado ou desligado | `thinking_level`, com os valores `minimal`, `low`, `medium` e `high` |
| Texto do raciocínio interno | `reasoning=True` grava em `additional_kwargs["reasoning_content"]` | `include_thoughts=True` grava em um bloco de `content_blocks` com `type` igual a `reasoning` |
| Campo `content` da resposta | uma string | uma lista de blocos |

O terceiro e o quarto ponto da tabela têm consequência direta nesta aula. A amostragem fixa do Gemini retira o controle usado pela seção 5 para gerar cadeias diferentes, e a variação passa a vir da amostragem que o próprio modelo aplica. O acesso ao texto do raciocínio interno tem parâmetro e destino diferentes nos dois provedores, e a função `raciocinio_interno` da seção 0.5 cobre os dois casos.

O parâmetro `reasoning=False` retira o texto de raciocínio do `gpt-oss` do campo da resposta. Os tokens de raciocínio continuam sendo gerados e continuam contando no orçamento de `num_predict`. Quando o orçamento acaba antes da resposta visível, a chamada volta com `done_reason` igual a `length` e sem texto. Com o valor 2500 de `LIMITE_RESPOSTA`, nenhuma chamada das seções 1 a 8 voltou com o campo de texto vazio nas execuções gravadas neste arquivo.

As saídas gravadas neste arquivo vêm do Ollama. Para executar pelo Gemini, gere uma chave em [aistudio.google.com/apikey](https://aistudio.google.com/apikey), guarde-a na variável `GOOGLE_API_KEY` e troque o valor de `PROVEDOR`.

In [5]:
load_dotenv()

PROVEDOR = "ollama"  # troque para "gemini" depois de definir GOOGLE_API_KEY

MODELO_OLLAMA = "gpt-oss:120b"           # servido por https://ollama.com
MODELO_GEMINI = "gemini-3.5-flash-lite"  # servido pela API do Google AI Studio

URL_OLLAMA = "https://ollama.com"
TEMPERATURA = 0
LIMITE_RESPOSTA = 2500  # tokens gerados por chamada


def criar_modelo(temperatura: float = TEMPERATURA, expor_raciocinio: bool = False):
    """Instancia o cliente do provedor escolhido com o mesmo orçamento de geração."""
    if PROVEDOR == "ollama":
        chave = os.environ.get("OLLAMA_API_KEY")
        if not chave:
            raise RuntimeError(
                "OLLAMA_API_KEY não encontrada. Gere uma chave em "
                "https://ollama.com/settings/keys e defina a variável no arquivo .env, "
                "no ambiente do sistema ou na célula da seção 0.5."
            )

        return ChatOllama(
            model=MODELO_OLLAMA,
            base_url=URL_OLLAMA,
            client_kwargs={"headers": {"Authorization": f"Bearer {chave}"}},
            reasoning=expor_raciocinio,
            temperature=temperatura,
            num_predict=LIMITE_RESPOSTA,
        )

    if PROVEDOR == "gemini":
        chave = os.environ.get("GOOGLE_API_KEY")
        if not chave:
            raise RuntimeError(
                "GOOGLE_API_KEY não encontrada. Gere uma chave em "
                "https://aistudio.google.com/apikey e defina a variável no arquivo .env "
                "ou no ambiente do sistema."
            )

        # A família Gemini 3.x fixa a amostragem, e o argumento temperatura não é repassado.
        return ChatGoogleGenerativeAI(
            model=MODELO_GEMINI,
            api_key=chave,
            max_output_tokens=LIMITE_RESPOSTA,
            thinking_level="low",
            include_thoughts=expor_raciocinio,
        )

    raise RuntimeError(f"Provedor não reconhecido: {PROVEDOR}. Use 'ollama' ou 'gemini'.")


modelo = criar_modelo()

print(PROVEDOR, "|", type(modelo).__name__)

ollama | ChatOllama


### 0.6.1 Saída estruturada nos dois provedores

Das seções 3 a 8, o raciocínio chega em campos de um esquema Pydantic. Dois pontos da chamada `with_structured_output` dependem do provedor e ficam concentrados nas duas funções abaixo.

O primeiro é o argumento `method`. O valor padrão do `ChatOllama` pede o esquema JSON diretamente ao modelo, e o `gpt-oss:120b` responde com a prosa dos passos em vez do objeto, o que termina em `OutputParserException` (3 execuções de 3). Com `method="function_calling"`, o esquema é declarado como ferramenta e o modelo preenche os argumentos dela, forma que os dois provedores aceitam. O notebook usa esse valor nos dois casos.

O segundo é o retorno vazio. O parser devolve `None` quando a resposta não traz a chamada com os campos preenchidos, e o acesso ao atributo seguinte quebra com `AttributeError` em vez de levantar um erro que nomeie a causa. `invocar_estruturado` repete o pedido quando isso acontece.

In [6]:
def criar_estruturado(esquema: type[BaseModel], temperatura: float = TEMPERATURA):
    """Liga um esquema Pydantic ao modelo pelo método que os dois provedores aceitam."""
    return criar_modelo(temperatura).with_structured_output(esquema, method="function_calling")


def invocar_estruturado(estruturado, mensagens: list[AnyMessage], tentativas: int = 3):
    """Repete o pedido enquanto o parser devolver None, e levanta erro ao esgotar as tentativas."""
    for _ in range(tentativas):
        resultado = estruturado.invoke(mensagens)

        if resultado is not None:
            return resultado

    raise RuntimeError(f"O modelo não preencheu o esquema em {tentativas} tentativas.")

A célula abaixo grava o problema usado por todas as seções. Ele é uma string única, reenviada em cada chamada como mensagem do usuário.

In [7]:
PROBLEMA = """Renan ainda não começou os módulos A5, A6, A7, A8, A9 e A10, que têm,
respectivamente, 4, 5, 4, 3, 6 e 5 aulas. Cada aula dura 50 minutos, e uma aula não pode
ser dividida entre dois dias. Faltam 10 dias para a prova final, e Renan tem 140 minutos
livres por dia. Ele termina o conteúdo a tempo?"""

print(PROBLEMA)

Renan ainda não começou os módulos A5, A6, A7, A8, A9 e A10, que têm,
respectivamente, 4, 5, 4, 3, 6 e 5 aulas. Cada aula dura 50 minutos, e uma aula não pode
ser dividida entre dois dias. Faltam 10 dias para a prova final, e Renan tem 140 minutos
livres por dia. Ele termina o conteúdo a tempo?


---

# 1. Raciocínio interno e raciocínio na resposta

Um modelo de raciocínio gera duas sequências de tokens por chamada. A primeira é o raciocínio interno, produzida antes da resposta e mantida fora do campo `content`. A segunda é a resposta, que a aplicação recebe. As duas são cobradas e as duas ocupam o orçamento de geração; só a segunda aparece por padrão.

A célula abaixo pede a conclusão em uma palavra, sem nenhuma técnica de raciocínio aplicada ao prompt. A contagem de tokens de saída é impressa ao lado do texto devolvido, para comparar o tamanho das duas sequências.

In [8]:
resposta_direta = modelo.invoke([
    SystemMessage("Responda apenas com 'sim' ou 'não', sem explicação."),
    HumanMessage(PROBLEMA),
])

print("texto da resposta:", repr(resposta_direta.text))
print("tokens de saída:", resposta_direta.usage_metadata["output_tokens"])
print("raciocínio recuperado:", repr(raciocinio_interno(resposta_direta)))

texto da resposta: 'não'
tokens de saída: 354
raciocínio recuperado: ''


## 1.1 Leitura da execução

O texto devolvido tem uma palavra, e a contagem de tokens de saída da mesma chamada, impressa ao lado, é de algumas centenas. A diferença entre os dois números é o raciocínio interno, gerado e cobrado nesta chamada. A função `raciocinio_interno` devolve string vazia porque `criar_modelo` foi chamada com `expor_raciocinio=False`, que é o valor padrão.

Duas consequências para a aplicação. A primeira é de orçamento: o limite de `LIMITE_RESPOSTA` é dividido entre o raciocínio e a resposta, e um limite apertado consome os tokens no raciocínio e devolve o campo de texto vazio. A segunda é de auditoria: com a conclusão em uma palavra, não há no texto da resposta por onde verificar qual das duas contas da seção 0.4 produziu o resultado.

## 1.2 Raciocínio interno recuperado do provedor

Os dois provedores devolvem o texto do raciocínio interno sob um parâmetro próprio, e `criar_modelo` liga os dois pelo argumento `expor_raciocinio`. A célula abaixo repete a pergunta com esse argumento ligado.

In [9]:
modelo_com_raciocinio = criar_modelo(expor_raciocinio=True)

resposta_exposta = modelo_com_raciocinio.invoke([
    SystemMessage("Responda apenas com 'sim' ou 'não', sem explicação."),
    HumanMessage(PROBLEMA),
])

texto_interno = raciocinio_interno(resposta_exposta)

print("texto da resposta:", repr(resposta_exposta.text))
print("caracteres de raciocínio interno:", len(texto_interno))
print("-" * 70)
print(texto_interno[:900])

texto da resposta: 'não'
caracteres de raciocínio interno: 1004
----------------------------------------------------------------------
The user asks a question in Portuguese: "Renan ainda não começou os módulos A5, A6, A7, A8, A9 e A10, que têm, respectivamente, 4, 5, 4, 3, 6 e 5 aulas. Cada aula dura 50 minutos, e uma aula não pode ser dividida entre dois dias. Faltam 10 dias para a prova final, e Renan tem 140 minutos livres por dia. Ele termina o conteúdo a tempo?" The user wants a yes/no answer. The developer instruction says: "Responda apenas com 'sim' ou 'não', sem explicação." So we must answer only "sim" or "não". Let's compute: total number of classes: 4+5+4+3+6+5 = 27 classes. Each class 50 minutes => total minutes = 27*50 = 1350 minutes. He has 10 days * 140 minutes = 1400 minutes total. However constraint: each class cannot be split across days, but each class is 50 minutes, and daily capacity is 140 minutes. So per day he can fit at most floor(140/50) = 2 classes (since 2*

## 1.3 Discussão

O raciocínio interno recuperado acima é um registro do que o modelo gerou. O raciocínio que as próximas seções produzem vem de um campo pedido pela aplicação, e quatro propriedades separam os dois:

| Propriedade | Raciocínio interno | Raciocínio na resposta |
|---|---|---|
| Origem | canal do provedor, ligado por parâmetro | campo do esquema pedido no prompt |
| Formato | prosa livre, no idioma escolhido pelo modelo | o que o esquema declarar |
| Acesso pelo código | um campo por provedor, com nomes diferentes | o mesmo atributo em qualquer provedor |
| Presença | depende do modelo ter modo de raciocínio | independe do modo de raciocínio do modelo |

O texto impresso na subseção 1.2 saiu em inglês, com a pergunta escrita em português. Um provedor pode também devolver um resumo do raciocínio em vez do texto gerado, ou não devolver nada. As sete técnicas seguintes não dependem desse canal: elas pedem os passos dentro da resposta, no idioma e no formato declarados pela aplicação.

---

# 2. Few-Shot Prompting

Few-Shot Prompting descreve a tarefa por exemplos completos de entrada e saída, escritos no prompt, em lugar de uma instrução sobre o que fazer. O modelo recebe o par pergunta e resposta já resolvido e produz a resposta nova no mesmo formato.

A técnica atua sobre o formato da saída. Os passos de raciocínio aparecem na resposta quando os exemplos os trazem, e ficam de fora quando os exemplos mostram apenas a conclusão. As duas subseções abaixo mudam somente os exemplos, e mantêm a instrução e a pergunta.

In [10]:
EXEMPLOS_SEM_RACIOCINIO = """Pergunta: Uma aluna tem 6 aulas de 40 minutos e 3 dias, com 80 minutos livres por dia. Ela termina?
Resposta: Sim.

Pergunta: Um aluno tem 10 aulas de 50 minutos e 2 dias, com 150 minutos livres por dia. Ele termina?
Resposta: Não."""

resposta_sem_raciocinio = modelo.invoke([
    SystemMessage(f"Siga o padrão dos exemplos abaixo.\n\n{EXEMPLOS_SEM_RACIOCINIO}"),
    HumanMessage(f"Pergunta: {PROBLEMA}"),
])

print(resposta_sem_raciocinio.text)

Resposta: Não.


## 2.1 Exemplos com os passos escritos

Os exemplos abaixo acrescentam um campo `Raciocínio:` antes da conclusão, e resolvem o mesmo par de perguntas da célula anterior pelo caminho das aulas inteiras. A instrução do sistema continua com uma frase.

In [11]:
EXEMPLOS_COM_RACIOCINIO = """Pergunta: Uma aluna tem 6 aulas de 40 minutos e 3 dias, com 80 minutos livres por dia. Ela termina?
Raciocínio: Em 80 minutos cabem 2 aulas inteiras de 40 minutos. Em 3 dias, são 6 aulas. Ela precisa de 6.
Resposta: Sim.

Pergunta: Um aluno tem 10 aulas de 50 minutos e 2 dias, com 150 minutos livres por dia. Ele termina?
Raciocínio: Em 150 minutos cabem 3 aulas inteiras de 50 minutos. Em 2 dias, são 6 aulas. Ele precisa de 10, e 6 é menor que 10.
Resposta: Não."""

resposta_com_raciocinio = modelo.invoke([
    SystemMessage(f"Siga o padrão dos exemplos abaixo.\n\n{EXEMPLOS_COM_RACIOCINIO}"),
    HumanMessage(f"Pergunta: {PROBLEMA}"),
])

print(resposta_com_raciocinio.text)

**Pergunta:** Renan ainda não começou os módulos A5, A6, A7, A8, A9 e A10, que têm, respectivamente, 4, 5, 4, 3, 6 e 5 aulas. Cada aula dura 50 minutos, e uma aula não pode ser dividida entre dois dias. Faltam 10 dias para a prova final, e Renan tem 140 minutos livres por dia. Ele termina o conteúdo a tempo?  

**Raciocínio:**  
- Total de aulas necessárias = 4 + 5 + 4 + 3 + 6 + 5 = **27 aulas**.  
- Cada aula tem 50 min. Em 140 min disponíveis por dia cabem, no máximo, **2 aulas completas** (2 × 50 = 100 min). O restante (40 min) não serve para outra aula inteira.  
- Em 10 dias, Renan pode assistir a **2 × 10 = 20 aulas**.  
- Ele precisa de 27 aulas, mas só consegue 20. Como 20 < 27, ele não conseguirá terminar a tempo.  

**Resposta:** Não.


## 2.2 Discussão

As duas células diferem nos exemplos e produzem saídas de formatos diferentes. A primeira devolve a conclusão, e a segunda devolve os passos e a conclusão, com os rótulos que os exemplos usaram. Nenhuma das duas instruções de sistema descreve a tarefa ou pede raciocínio.

Os exemplos carregam também o método de cálculo. Os dois exemplos da subseção 2.1 convertem minutos livres em aulas inteiras antes de contar, que é o caminho correto da tabela da seção 0.4, e o raciocínio da resposta refaz essa conversão sobre os números novos.

A técnica não fornece um formato que o código possa ler. A resposta é uma string, e separar os passos da conclusão exige localizar os rótulos `Raciocínio:` e `Resposta:` no texto. Na execução gravada eles vieram em negrito, `**Raciocínio:**`, precedidos da pergunta repetida, formato que nenhum dos dois exemplos usou. Uma leitura que procure o rótulo no início da linha não o encontra, e a chamada não acusa nada. A seção 3 pede os mesmos dois campos em um esquema, e a resposta chega como objeto.

---

# 3. Chain-of-Thought

Chain-of-Thought pede os passos do cálculo antes da conclusão, na mesma chamada. A instrução no prompt é uma das formas de obter isso; a outra é declarar um esquema com um campo para os passos e outro para a conclusão, e deixar o modelo preencher os dois.

A segunda forma é a usada aqui. O esquema abaixo tem três campos: o texto dos passos, um veredito em booleano e a frase de conclusão. O booleano existe para que o código decida sem interpretar texto, e é o campo que a votação da seção 5 conta.

In [12]:
class RespostaComRaciocinio(BaseModel):
    """Passos do cálculo, veredito e conclusão em uma frase."""

    raciocinio: str = Field(description="Passos do cálculo, numerados, um por linha.")
    termina_a_tempo: bool = Field(description="true se o aluno termina no prazo, false se não termina.")
    conclusao: str = Field(description="Uma frase com a conclusão e o número que a sustenta.")


cot = criar_estruturado(RespostaComRaciocinio)

resultado_cot = invocar_estruturado(cot, [
    SystemMessage(
        "Resolva o problema passo a passo antes de concluir. "
        "Preencha os campos raciocinio, termina_a_tempo e conclusao."
    ),
    HumanMessage(PROBLEMA),
])

mostrar_raciocinio(resultado_cot.raciocinio, resultado_cot.conclusao)
print("-" * 70)
print("termina_a_tempo:", resultado_cot.termina_a_tempo)

RACIOCÍNIO
1. Total de aulas restantes = 4+5+4+3+6+5 = 27 aulas.
2. Cada aula dura 50 minutos, total de minutos necessários = 27 × 50 = 1350 minutos.
3. Tempo livre total nos 10 dias = 10 × 140 = 1400 minutos, portanto há minutos suficientes em soma.
4. Como uma aula não pode ser dividida, o número máximo de aulas que cabem em um dia é floor(140/50) = 2 aulas por dia.
5. Em 10 dias, o número máximo de aulas que Renan pode assistir = 10 × 2 = 20 aulas.
6. Como 27 aulas > 20 aulas, ele não consegue encaixar todas as aulas nos dias disponíveis.
----------------------------------------------------------------------
CONCLUSÃO
Renan não termina o conteúdo a tempo.
----------------------------------------------------------------------
termina_a_tempo: False


## 3.1 Leitura da execução

A chamada devolveu um objeto `RespostaComRaciocinio`, e os três campos vieram preenchidos. Os passos numerados estão em `raciocinio`, a conclusão em linguagem natural está em `conclusao`, e a decisão que o código usa está em `termina_a_tempo`, como booleano.

Os passos gravados percorrem os dois caminhos da tabela da seção 0.4: a soma em minutos aparece no meio da cadeia, e o passo seguinte a descarta pela indivisibilidade da aula antes de decidir. O campo `termina_a_tempo` recebeu `False`, que é a conclusão correta.

## 3.2 Discussão

O custo desta técnica sobre a resposta direta da seção 1 é o tamanho da saída. São os mesmos tokens de entrada, uma chamada de modelo, e mais tokens gerados, porque os passos agora fazem parte da resposta. Nenhuma chamada adicional é feita.

Os campos separados mudam o que a aplicação pode fazer com o resultado. A interface exibe `conclusao`, o registro de depuração guarda `raciocinio`, e o roteamento do programa lê `termina_a_tempo`, sem que nenhum dos três precise ser extraído do texto dos outros. As técnicas das seções 4 a 8 mantêm essa separação e mudam a forma do campo de raciocínio.

---

# 4. Self-Ask

Self-Ask dá forma de perguntas ao raciocínio. Em vez de um texto contínuo, o modelo escreve as subperguntas que precisa responder para chegar à conclusão, responde cada uma, e conclui a partir dessas respostas.

Fluxo:

```text
pergunta original
  ↓
subpergunta 1 → resposta intermediária 1
  ↓
subpergunta 2 → resposta intermediária 2
  ↓
subpergunta n → resposta intermediária n
  ↓
conclusão
```

O esquema abaixo tem uma lista de objetos, cada um com a subpergunta e a resposta dela. É um esquema aninhado, e a instrução do sistema nomeia as chaves do objeto interno, pelo motivo registrado na subseção 4.2.

In [13]:
class ParPerguntaResposta(BaseModel):
    """Uma subpergunta e a resposta dela."""

    subpergunta: str = Field(description="Texto da subpergunta.")
    resposta_intermediaria: str = Field(description="Texto da resposta, sempre uma frase.")


class RespostaSelfAsk(BaseModel):
    """Cadeia de subperguntas respondidas, veredito e conclusão."""

    subperguntas: list[ParPerguntaResposta] = Field(
        description="Lista de objetos, cada um com as chaves subpergunta e resposta_intermediaria."
    )
    termina_a_tempo: bool = Field(description="true se o aluno termina no prazo, false se não termina.")
    conclusao: str = Field(description="Uma frase com a conclusão.")


self_ask = criar_estruturado(RespostaSelfAsk)

resultado_self_ask = invocar_estruturado(self_ask, [
    SystemMessage(
        "Responda decompondo a pergunta em subperguntas, na ordem em que precisam ser "
        "respondidas, e respondendo cada uma antes de concluir. Preencha os campos: "
        "subperguntas (lista de objetos com as chaves subpergunta e resposta_intermediaria, "
        "ambas em texto), termina_a_tempo e conclusao."
    ),
    HumanMessage(PROBLEMA),
])

raciocinio_self_ask = "\n".join(
    f"{indice}. {par.subpergunta}\n   {par.resposta_intermediaria}"
    for indice, par in enumerate(resultado_self_ask.subperguntas, start=1)
)

mostrar_raciocinio(raciocinio_self_ask, resultado_self_ask.conclusao)

RACIOCÍNIO
1. Quantas aulas no total dos módulos A5 a A10?
   4 + 5 + 4 + 3 + 6 + 5 = 27 aulas.
2. Quantos minutos são necessários no total?
   27 aulas × 50 minutos = 1350 minutos.
3. Quantos minutos Renan tem disponíveis nos 10 dias?
   10 dias × 140 minutos/dia = 1400 minutos.
4. Quantas aulas completas cabem em um dia?
   Cada aula tem 50 minutos; 140/50 = 2 aulas (100 minutos) como máximo, pois 3 aulas exigiriam 150 minutos > 140.
5. Quantos dias são necessários para completar 27 aulas com no máximo 2 aulas por dia?
   27 aulas ÷ 2 aulas/dia = 13,5 dias → são necessários 14 dias.
----------------------------------------------------------------------
CONCLUSÃO
Renan não termina o conteúdo a tempo.


## 4.1 Leitura da execução

O campo `subperguntas` veio como lista, e cada item tem as duas chaves declaradas. A sequência impressa tem cinco itens: somar as aulas, convertê-las em minutos, somar os minutos livres dos 10 dias, contar quantas aulas inteiras cabem em um dia, e dividir as 27 aulas pelas 2 por dia, o que dá os 14 dias da conclusão. Os dois caminhos da tabela da seção 0.4 aparecem na mesma lista, e o item que decide a pergunta é o último.

Cada item é um ponto de verificação isolado. Uma soma errada na primeira subpergunta fica visível como um valor errado naquele item, e as respostas seguintes que dependem dele podem ser conferidas contra esse valor. No campo único da seção 3, o mesmo erro está dentro de um texto corrido.

## 4.2 Discussão

O esquema desta seção é aninhado, e é o ponto em que o `gpt-oss:120b` diverge do `gemini-3.5-flash-lite`. Com a instrução do sistema descrevendo apenas a tarefa, o `gpt-oss` preencheu `subperguntas` com uma lista de strings, e a validação do Pydantic recusou os itens (4 execuções de 4). Com a linha que nomeia as chaves do objeto interno, a lista veio com os objetos (5 execuções de 5). O `gemini-3.5-flash-lite` produziu a lista de objetos nas duas versões.

A descrição do campo no `Field` e a instrução no prompt chegam ao modelo por caminhos diferentes: a primeira dentro da declaração do esquema, a segunda no texto da conversa. Repetir o formato nos dois sustenta a execução com os dois provedores.

A estrutura em subperguntas abre um caminho que a seção 3 não tem: a resposta de uma subpergunta pode vir de outra fonte. Uma subpergunta como a contagem de aulas é uma soma, e a seção 6 a entrega a uma função Python em vez de deixar o modelo calculá-la.

---

# 5. Self-Consistency

Self-Consistency chama o modelo várias vezes com a mesma pergunta, coleta uma cadeia de raciocínio por chamada e decide por votação entre os vereditos. A premissa é que caminhos gerados de forma independente erram em pontos diferentes, e o veredito repetido pela maioria conta com mais caminhos independentes a sustentá-lo.

```text
cadeia 1 → veredito
cadeia 2 → veredito     contagem dos vereditos
cadeia 3 → veredito  →  ────────────────────  →  veredito majoritário
cadeia 4 → veredito         Counter
cadeia 5 → veredito
```

A variação entre as chamadas vem da amostragem. No Ollama, `temperature` controla essa variação, e a função abaixo recebe o valor por parâmetro. No Gemini, a amostragem é fixa e o argumento não é repassado, o que deixa a variação por conta do próprio modelo.

In [14]:
def self_consistency(pergunta: str, amostras: int = 5, temperatura: float = 1.0) -> dict:
    """Amostra cadeias independentes para a mesma pergunta e conta os vereditos."""
    cot_amostrado = criar_estruturado(RespostaComRaciocinio, temperatura=temperatura)

    cadeias = [
        invocar_estruturado(cot_amostrado, [
            SystemMessage(
                "Resolva o problema passo a passo antes de concluir. "
                "Preencha os campos raciocinio, termina_a_tempo e conclusao."
            ),
            HumanMessage(pergunta),
        ])
        for _ in range(amostras)
    ]

    contagem = Counter(cadeia.termina_a_tempo for cadeia in cadeias)
    veredito, votos = contagem.most_common(1)[0]

    return {"cadeias": cadeias, "contagem": contagem, "veredito": veredito, "votos": votos}

## 5.1 Execução com cinco amostras

A célula abaixo faz cinco chamadas e imprime a cadeia de cada uma antes da contagem. As cinco cadeias são impressas, incluindo as que perderam a votação: a divergência entre elas é a informação que a técnica produz.

In [15]:
resultado_sc = self_consistency(PROBLEMA, amostras=5)

for indice, cadeia in enumerate(resultado_sc["cadeias"], start=1):
    print(f"=== cadeia {indice} | termina_a_tempo={cadeia.termina_a_tempo} ===")
    print(cadeia.raciocinio)
    print()

print("-" * 70)
print("contagem dos vereditos:", dict(resultado_sc["contagem"]))
print(f"veredito majoritário: {resultado_sc['veredito']} com {resultado_sc['votos']} de 5 votos")

=== cadeia 1 | termina_a_tempo=False ===
1. Total de aulas = 4+5+4+3+6+5 = 27.
2. Cada aula = 50 min, portanto minutos totais necessários = 27*50 = 1350.
3. Tempo disponível total = 10 dias * 140 min/dia = 1400 min.
4. Por dia, número máximo de aulas inteiras = floor(140/50) = 2 aulas.
5. Total máximo de aulas que pode fazer em 10 dias = 2 * 10 = 20 aulas.
6. Como 20 < 27, ele não consegue concluir todas as aulas a tempo.

=== cadeia 2 | termina_a_tempo=False ===
1. Total de aulas necessárias = 4+5+4+3+6+5 = 27 aulas.
2. Cada aula dura 50 minutos, total de minutos necessários = 27 * 50 = 1350 minutos.
3. Tempo total disponível = 10 dias * 140 minutos/dia = 1400 minutos.
4. Como 1400 > 1350, em termos de minutos parece suficiente.
5. Contudo, uma aula não pode ser dividida e cabe no máximo floor(140/50) = 2 aulas por dia (2*50=100 ≤ 140, 3*50=150 > 140).
6. Em 10 dias, número máximo de aulas que ele pode assistir = 10 * 2 = 20 aulas.
7. Como 20 aulas < 27 aulas necessárias, ele não cons

## 5.2 Leitura da execução

As cinco cadeias são textos diferentes entre si, com a mesma pergunta e o mesmo prompt. Elas diferem no número de passos, entre 4 e 7, na ordem das contas e na presença do total em minutos.

Os dois vereditos possíveis correspondem aos dois caminhos da tabela da seção 0.4: `False` vem da contagem por aulas inteiras, e `True` vem da comparação direta entre 1350 e 1400 minutos, que ignora a restrição de não dividir uma aula. Na execução gravada, as cinco cadeias chegaram a `False`, e a votação fechou em 5 a 0. Quatro delas somam os 1350 minutos e descartam esse caminho no passo seguinte, pela contagem de aulas inteiras por dia; a restante não calcula o total em minutos e vai das 2 aulas por dia direto aos 14 dias necessários.

A unanimidade é um resultado desta combinação de modelo e problema. Em 17 amostras do mesmo problema pelo `gemini-3.5-flash-lite`, o caminho por minutos decidiu a resposta em 4 delas, e uma execução desta mesma célula com aquele modelo fechou a votação em 4 a 1.

A votação não verifica o cálculo. Ela conta quantas cadeias chegaram a cada veredito, e um erro cometido pela maioria das cadeias sai da contagem como veredito vencedor.

## 5.3 Discussão

O custo é o número de amostras vezes o custo de uma chamada com Chain-of-Thought. A técnica troca esse custo por uma medida que a chamada única não tem: a proporção de votos. Uma votação de 5 a 0 e uma de 3 a 2 devolvem o mesmo veredito, e a segunda registra que o problema separou as cadeias.

Essa proporção é usável pela aplicação. O programa aceita o veredito quando os votos passam de um limite, e encaminha para verificação humana quando ficam abaixo dele.

---

# 6. ReAct

ReAct alterna dois tipos de passo na mesma conversa: um passo de raciocínio, escrito pelo modelo, e um passo de ação, que a aplicação executa e cujo resultado volta como observação. O raciocínio desta técnica é o próprio histórico de mensagens que o laço acumula.

A ferramenta abaixo converte um total de aulas em dias, com o arredondamento para cima feito em Python. A conta sai do modelo e passa a ser um dado observado.

```text
pergunta
  ↓
AIMessage com raciocínio e pedido de chamada
  ↓
aplicação executa a função
  ↓
ToolMessage com o resultado        ← observação
  ↓
AIMessage com a conclusão
```

In [16]:
@tool(parse_docstring=True)
def calcular_dias_necessarios(total_aulas: int, aulas_por_dia: int) -> str:
    """Calcula quantos dias são necessários para assistir a um total de aulas.

    Args:
        total_aulas: número de aulas a assistir.
        aulas_por_dia: número de aulas inteiras que cabem em um dia.
    """
    dias = math.ceil(total_aulas / aulas_por_dia)

    return json.dumps(
        {"total_aulas": total_aulas, "aulas_por_dia": aulas_por_dia, "dias_necessarios": dias},
        ensure_ascii=False,
    )


PROMPT_REACT = """Resolva o problema em turnos. Em cada turno, escreva no texto da resposta o
raciocínio do passo antes de pedir a ferramenta.
Use a ferramenta calcular_dias_necessarios para converter aulas em dias; não faça essa conta
de cabeça. Responda em português."""


def laco_react(modelo_base, pergunta: str, limite_turnos: int = 4) -> list[AnyMessage]:
    """Alterna chamadas ao modelo e execuções da ferramenta até a resposta sem pedidos."""
    modelo_com_ferramenta = modelo_base.bind_tools([calcular_dias_necessarios])
    mensagens: list[AnyMessage] = [SystemMessage(PROMPT_REACT), HumanMessage(pergunta)]

    for _ in range(limite_turnos):
        resposta = modelo_com_ferramenta.invoke(mensagens)
        mensagens.append(resposta)

        if not resposta.tool_calls:
            break

        for chamada in resposta.tool_calls:
            observacao = calcular_dias_necessarios.invoke(chamada["args"])
            mensagens.append(ToolMessage(content=observacao, tool_call_id=chamada["id"]))

    return mensagens


mensagens_react = laco_react(modelo, PROBLEMA)

mostrar_conversa(mensagens_react)

print("CONCLUSÃO")
print(mensagens_react[-1].text)

[0] SystemMessage
    texto: Resolva o problema em turnos. Em cada turno, escreva no texto da resposta o
raciocínio do passo antes de pedir a ferramenta.
Use a ferramenta calcular_dias_necessarios para converter aulas em dias; não faça essa conta
de cabeça. Responda em português.
----------------------------------------------------------------------
[1] HumanMessage
    texto: Renan ainda não começou os módulos A5, A6, A7, A8, A9 e A10, que têm,
respectivamente, 4, 5, 4, 3, 6 e 5 aulas. Cada aula dura 50 minutos, e uma aula não pode
ser dividida entre dois dias. Faltam 10 dias para a prova final, e Renan tem 140 minutos
livres por dia. Ele termina o conteúdo a tempo?
----------------------------------------------------------------------
[2] AIMessage
    pedido: calcular_dias_necessarios({'aulas_por_dia': 2, 'total_aulas': 27})
----------------------------------------------------------------------
[3] ToolMessage
    resultado: {"total_aulas": 27, "aulas_por_dia": 2, "dias_necessarios"

## 6.1 Leitura da execução

O histórico impresso tem a instrução de sistema, a pergunta, a resposta com o pedido de chamada, a observação da ferramenta e a resposta final. Os argumentos do pedido, `total_aulas` e `aulas_por_dia`, foram deduzidos do enunciado, e o valor de `dias_necessarios` veio de `math.ceil`, não do modelo.

O texto da `AIMessage` que carrega o pedido saiu vazio na execução gravada, apesar de o prompt pedir o raciocínio antes da ação. O `gpt-oss:120b` escreveu esse raciocínio no canal interno da seção 1, e ele fica fora de `content`. Um modelo criado com `expor_raciocinio=True` recupera esse texto por `raciocinio_interno`, e o exercício 3 pede essa execução. Nas execuções pelo `gemini-3.5-flash-lite`, o mesmo prompt produziu o raciocínio dentro de `content`, antes do pedido.

## 6.2 Discussão

O raciocínio do ReAct tem uma propriedade que as técnicas das seções 3 a 5 não têm: parte dele são dados de fora do modelo. A `ToolMessage` com `dias_necessarios` igual a 14 é um valor calculado em Python, e a resposta final parte dele em vez de um número escrito pelo modelo.

A técnica não resolve o formato sozinha. O histórico é uma lista de mensagens de texto, e recuperar a conclusão dele significa ler a última `AIMessage`. As duas formas se combinam: o laço executa as ações, e a última chamada usa um esquema como o da seção 3 para devolver a conclusão em campos.

---

# 7. Least-to-Most

Least-to-Most separa a decomposição da resolução em duas etapas. A primeira chamada lista os subproblemas, do mais simples ao mais complexo, sem resolver nenhum. As chamadas seguintes resolvem um subproblema por vez, e cada uma recebe as respostas dos subproblemas anteriores.

A ordem é a restrição da técnica: cada subproblema depende apenas dos que vieram antes. No Self-Ask da seção 4, as subperguntas e as respostas saem juntas de uma chamada, e nada separa a lista de perguntas do ato de respondê-las.

```text
chamada 1: decompor  →  [subproblema 1, subproblema 2, subproblema 3]
                              ↓
chamada 2: resolver 1   (contexto: nenhum)
chamada 3: resolver 2   (contexto: resposta 1)
chamada 4: resolver 3   (contexto: respostas 1 e 2)
                              ↓
chamada 5: concluir     (contexto: todas as respostas)
```

In [17]:
class Decomposicao(BaseModel):
    """Lista ordenada de subproblemas."""

    subproblemas: list[str] = Field(
        description="Subproblemas em ordem, do mais simples ao mais complexo, um texto por item."
    )


INSTRUCAO_DECOMPOSICAO = (
    "Decomponha o problema em no máximo 4 subproblemas, do mais simples ao mais "
    "complexo, cada um resolvível com os resultados dos anteriores. "
    "Preencha o campo subproblemas e não resolva nenhum deles."
)


def least_to_most(pergunta: str, instrucao: str = INSTRUCAO_DECOMPOSICAO) -> dict:
    """Decompõe o problema e resolve um subproblema por chamada, acumulando as respostas."""
    decomposicao = invocar_estruturado(criar_estruturado(Decomposicao), [
        SystemMessage(instrucao),
        HumanMessage(pergunta),
    ])

    respostas = []

    for subproblema in decomposicao.subproblemas:
        contexto = "\n".join(
            f"- {texto}: {resposta}"
            for texto, resposta in zip(decomposicao.subproblemas, respostas)
        ) or "nenhum subproblema resolvido ainda"

        resposta = modelo.invoke([
            SystemMessage("Resolva apenas o subproblema atual, em uma frase, usando o contexto."),
            HumanMessage(
                f"Problema: {pergunta}\n\nSubproblemas já resolvidos:\n{contexto}\n\n"
                f"Subproblema atual: {subproblema}"
            ),
        ])

        respostas.append(resposta.text.strip())

    final = invocar_estruturado(criar_estruturado(RespostaComRaciocinio), [
        SystemMessage(
            "Conclua a partir dos subproblemas resolvidos. Preencha os campos raciocinio, "
            "termina_a_tempo e conclusao."
        ),
        HumanMessage(
            f"Problema: {pergunta}\n\n"
            + "\n".join(
                f"- {texto}: {resposta}"
                for texto, resposta in zip(decomposicao.subproblemas, respostas)
            )
        ),
    ])

    return {"subproblemas": decomposicao.subproblemas, "respostas": respostas, "final": final}

## 7.1 Execução das duas etapas

A célula abaixo chama `least_to_most` sobre `PROBLEMA` e imprime cada subproblema ao lado da resposta que ele recebeu. A contagem final soma a chamada de decomposição, uma chamada por subproblema e a chamada de conclusão.

In [18]:
resultado_ltm = least_to_most(PROBLEMA)

raciocinio_ltm = "\n".join(
    f"{indice}. {subproblema}\n   {resposta}"
    for indice, (subproblema, resposta) in enumerate(
        zip(resultado_ltm["subproblemas"], resultado_ltm["respostas"]), start=1
    )
)

mostrar_raciocinio(raciocinio_ltm, resultado_ltm["final"].conclusao)
print("-" * 70)
print("chamadas de modelo:", 1 + len(resultado_ltm["subproblemas"]) + 1)

RACIOCÍNIO
1. Calcular o número total de aulas que Renan precisa assistir nos módulos A5 a A10.
   Renan precisa assistir a um total de **27 aulas** nos módulos A5 a A10.
2. Calcular o tempo total em minutos necessário para concluir todas essas aulas (número total de aulas multiplicado por 50 minutos).
   Renan precisará de **1 350 minutos** (27 aulas × 50 min por aula) para concluir todos os módulos A5 a A10.
3. Calcular o tempo total disponível de estudo nos 10 dias restantes (140 minutos por dia multiplicado por 10 dias).
   O tempo total disponível nos 10 dias é de **1 400 minutos** (140 min × 10 dias).
4. Verificar se é possível distribuir as aulas nos 10 dias, considerando que cada dia pode comportar no máximo duas aulas de 50 minutos (pois três aulas excederiam os 140 minutos disponíveis).
   Não, porque em 10 dias com no máximo duas aulas por dia só dá 20 aulas (10 × 2), e Renan precisa assistir 27 aulas.
----------------------------------------------------------------------
CO

## 7.2 Leitura da execução

A lista de subproblemas saiu da primeira chamada, e cada resposta impressa saiu de uma chamada própria. A ordem produzida começa pela contagem de aulas, que não depende de nenhum outro resultado, e termina na comparação que decide a pergunta, que depende dos anteriores. O contexto entregue em cada chamada cresce a cada passo: a primeira recebe a linha `nenhum subproblema resolvido ainda`, e a última recebe a lista das respostas já produzidas.

Na execução gravada, a lista cobre os dois caminhos da tabela da seção 0.4: o segundo e o terceiro subproblemas somam os minutos, e o quarto pergunta como distribuir as aulas no dia, com o limite de duas aulas de 50 minutos. A resposta do quarto decide a pergunta, e `termina_a_tempo` fica `False`, como nas seções 3 a 6 e 8.

A decomposição é feita antes de qualquer conta e fixa o caminho de raciocínio das chamadas seguintes. Cada uma resolve o subproblema que recebeu, e um subproblema ausente da lista não é recuperado nas etapas posteriores. O efeito é observável nas listas que param na comparação entre 1350 e 1400 minutos: em sete execuções desta célula medidas antes desta, cinco pelo `gpt-oss:120b` e duas pelo `gemini-3.5-flash-lite`, nenhum dos subproblemas tratou de quantas aulas inteiras cabem em um dia, e as sete conclusões saíram opostas à da seção 0.4, com cada chamada de resolução respondendo corretamente o subproblema que recebeu. As técnicas de uma chamada só, das seções 3 e 4, chegaram à indivisibilidade dentro da mesma cadeia em que somaram os minutos.

A lista muda entre execuções, porque sai de uma chamada de modelo. A subseção 7.3 acrescenta uma linha ao prompt de decomposição e repete as duas etapas.

## 7.3 Decomposição orientada

O parâmetro `instrucao` de `least_to_most` recebe o texto enviado à chamada de decomposição, e o valor padrão é `INSTRUCAO_DECOMPOSICAO`. A célula abaixo acrescenta uma frase a esse texto, nomeando a grandeza que decide o problema, e repete as duas etapas. As chamadas de resolução e a de conclusão não mudam.

In [19]:
instrucao_orientada = INSTRUCAO_DECOMPOSICAO + (
    " Inclua um subproblema sobre quantas aulas inteiras cabem em um dia."
)

resultado_orientado = least_to_most(PROBLEMA, instrucao=instrucao_orientada)

for indice, subproblema in enumerate(resultado_orientado["subproblemas"], start=1):
    print(f"{indice}. {subproblema}")

print("-" * 70)
print("conclusão:", resultado_orientado["final"].conclusao)
print("termina_a_tempo:", resultado_orientado["final"].termina_a_tempo)
print("na subseção 7.1:", resultado_ltm["final"].termina_a_tempo)

1. Quantas aulas inteiras de 50 minutos cabem nos 140 minutos livres de Renan por dia?
2. Qual é o número total de aulas que Renan ainda precisa assistir nos módulos A5 a A10?
3. Qual é o total de minutos necessários para concluir todas essas aulas?
4. Considerando o número máximo de aulas por dia (do subproblema 1) e o número total de aulas (do subproblema 2), Renan consegue concluir todas as aulas nos 10 dias disponíveis?
----------------------------------------------------------------------
conclusão: Renan não termina a tempo, pois precisa assistir 27 aulas e só consegue assistir 20 aulas nos 10 dias disponíveis.
termina_a_tempo: False
na subseção 7.1: False


## 7.4 Leitura da decomposição orientada

A lista devolvida traz, como primeiro item, um subproblema sobre quantas aulas inteiras de 50 minutos cabem em 140 minutos, e o item que decide a pergunta cita esse resultado ao lado do total de aulas. A grandeza que na subseção 7.1 apareceu no quarto item passa ao primeiro, e o total em minutos deixa de participar da decisão.

Os dois valores de `termina_a_tempo` impressos coincidem nesta execução. A frase acrescentada não contém a resposta nem o número de aulas por dia: ela nomeia a grandeza que precisa aparecer na decomposição, e as chamadas seguintes calculam o valor. O ponto de intervenção é a primeira das seis chamadas, e as outras cinco continuam iguais.

Quando a decomposição da subseção 7.1 omite essa grandeza, a mesma frase inverte a conclusão. As sete execuções medidas antes desta, nos dois provedores, registram esse caso: `termina_a_tempo` passou de `True` para `False` com o mesmo problema, o mesmo modelo e as mesmas chamadas de resolução.

## 7.5 Discussão

A decomposição em chamada separada distingue esta técnica do Self-Ask. Com a lista pronta antes de qualquer resolução, ela é um dado que a aplicação inspeciona: um subproblema pode ser descartado, reordenado, resolvido por uma ferramenta, ou submetido à votação da seção 5. Na seção 4, a lista e as respostas chegam juntas, e essa intervenção não tem onde acontecer.

A lista está disponível em `resultado_ltm["subproblemas"]` antes de qualquer resolução, e o que ela cobre é verificável nessas quatro strings: a presença de um subproblema sobre a indivisibilidade da aula separa as decomposições que decidem certo das que decidem errado. A subseção 7.3 age sobre esse ponto pelo prompt. O avaliador da seção 8 aplica uma verificação parecida ao mesmo caminho de cálculo e, na execução gravada, o reprova.

O preço é o número de chamadas, que cresce com o tamanho da decomposição. O limite de 4 subproblemas no prompt fixa o teto em 6 chamadas por pergunta, e a subseção 7.3 gasta essas 6 de novo para trocar uma linha da primeira.

---

# 8. Tree of Thoughts

As técnicas anteriores seguem um caminho de raciocínio por vez. Tree of Thoughts gera vários candidatos no mesmo ponto da resolução, avalia cada um por um critério declarado, e continua pelo mais bem avaliado.

O arranjo abaixo tem um nível: três abordagens propostas, uma nota para cada, e a resolução pela escolhida. A técnica completa repete geração e avaliação em cada nível e volta atrás quando um ramo é reprovado; o nível único mantém a estrutura de geração, avaliação e seleção com um número de chamadas fixo.

```text
                    problema
                       │
        ┌──────────────┼──────────────┐
   abordagem 1    abordagem 2    abordagem 3
        │              │              │
     nota 1         nota 2         nota 3      ← avaliação por critério
        └──────────────┼──────────────┘
                       ↓
              abordagem de maior nota
                       ↓
                   resolução
```

In [20]:
class Abordagens(BaseModel):
    """Formas distintas de decidir a pergunta."""

    abordagens: list[str] = Field(
        description="Descrições curtas de abordagens distintas, um texto por item."
    )


class Avaliacao(BaseModel):
    """Nota atribuída a uma abordagem."""

    nota: int = Field(description="Nota de 0 a 10 para a confiabilidade da abordagem.")
    justificativa: str = Field(description="Uma frase justificando a nota.")


CRITERIO = """Avalie a abordagem proposta para o problema. Considere se ela respeita todas as
restrições do enunciado. Preencha os campos nota e justificativa."""

## 8.1 Geração, avaliação e seleção

A célula abaixo percorre as três etapas em sequência. Uma chamada gera as abordagens candidatas, uma chamada por candidata atribui a nota e a justificativa, e `max` devolve o índice da maior nota.

In [21]:
candidatas = invocar_estruturado(criar_estruturado(Abordagens), [
    SystemMessage(
        "Proponha 3 abordagens distintas para decidir a pergunta, cada uma em uma frase. "
        "Abordagens distintas usam grandezas ou critérios diferentes. "
        "Preencha o campo abordagens e não resolva o problema."
    ),
    HumanMessage(PROBLEMA),
]).abordagens

avaliador = criar_estruturado(Avaliacao)
notas = []

for abordagem in candidatas:
    avaliacao = invocar_estruturado(avaliador, [
        SystemMessage(CRITERIO),
        HumanMessage(f"Problema: {PROBLEMA}\n\nAbordagem: {abordagem}"),
    ])
    notas.append(avaliacao)
    print(f"nota {avaliacao.nota:2d} | {abordagem}")
    print(f"         {avaliacao.justificativa}")
    print()

melhor = max(range(len(candidatas)), key=lambda indice: notas[indice].nota)

print("-" * 70)
print("abordagem escolhida:", candidatas[melhor])

nota  3 | Calcular o total de minutos necessários para todas as aulas e comparar com a capacidade total disponível nos 10 dias (140 min × 10).
         A abordagem soma apenas os minutos totais necessários e compara com a capacidade total de 10 dias (140 min × 10). Embora isso indique que 1350 min ≤ 1400 min, ela ignora a restrição de que cada aula tem 50 min e não pode ser dividida entre dias, o que limita a quantidade de aulas que cabem por dia a no máximo 2 (100 min). Assim, o método pode concluir erroneamente que Renan termina a tempo. Falta considerar o empacotamento diário, portanto a confiabilidade da abordagem é baixa.

nota  9 | Verificar, dia a dia, quantas aulas de 50 min podem ser encaixadas nos 140 min livres, acumulando até completar ou exceder o número total de aulas.
         A abordagem propõe percorrer os 10 dias, encaixando o maior número possível de aulas de 50 min em cada bloco de 140 min, respeitando a restrição de que a aula não pode ser dividida. Isso leva a cal

## 8.2 Resolução pela abordagem escolhida

A abordagem selecionada entra no prompt da chamada final, que devolve o esquema `RespostaComRaciocinio` da seção 3. A contagem impressa soma a chamada de geração, as três de avaliação e esta.

In [22]:
resultado_tot = invocar_estruturado(criar_estruturado(RespostaComRaciocinio), [
    SystemMessage(
        "Resolva o problema seguindo a abordagem indicada. "
        "Preencha os campos raciocinio, termina_a_tempo e conclusao."
    ),
    HumanMessage(f"Problema: {PROBLEMA}\n\nAbordagem a seguir: {candidatas[melhor]}"),
])

mostrar_raciocinio(resultado_tot.raciocinio, resultado_tot.conclusao)
print("-" * 70)
print("chamadas de modelo:", 1 + len(candidatas) + 1)

RACIOCÍNIO
1. Total de aulas = 4+5+4+3+6+5 = 27.
2. Cada aula = 50 min, total necessário = 27*50 = 1350 min.
3. Tempo livre diário = 140 min → número máximo de aulas por dia = floor(140/50) = 2.
4. Simulação dia a dia (10 dias): cada dia cabe 2 aulas → 2*10 = 20 aulas concluídas.
5. Aulas restantes após 10 dias = 27‑20 = 7, sem dias adicionais → não termina a tempo.
----------------------------------------------------------------------
CONCLUSÃO
Não, Renan não termina a tempo, pois precisa de 27 aulas e só consegue encaixar 20 nos 10 dias disponíveis.
----------------------------------------------------------------------
chamadas de modelo: 5


## 8.3 Leitura da execução

Na execução gravada, a primeira abordagem parte do total de minutos, a segunda percorre os dias encaixando aulas de 50 minutos nos 140 disponíveis, e a terceira propõe analisar a distribuição por módulo. A primeira é o caminho que a tabela da seção 0.4 marca como incorreto.

As notas saíram 3, 9 e 3, e cada justificativa nomeia o que falta à abordagem avaliada. As duas notas 3 vão para a que compara 1350 com 1400 minutos sem tratar da indivisibilidade, e para a que distribui as aulas por módulo sem chegar ao limite diário. A nota 9 vai para a que calcula as 2 aulas por dia e as confronta com as 27 necessárias. `max` devolveu o índice da segunda, e a subseção 8.2 resolveu por ela; entre notas iguais, `max` devolve o primeiro índice.

As abordagens geradas e as notas mudam entre execuções, porque saem de chamadas de modelo, e a nota decide a seleção. Em uma execução destas duas células pelo `gemini-3.5-flash-lite`, o avaliador deu a nota máxima à abordagem por minutos totais, a seleção seguiu por ela, e a subseção 8.2 concluiu que Renan termina a tempo.

## 8.4 Discussão

A diferença para o Self-Consistency da seção 5 está no que é gerado e em como se decide. Self-Consistency amostra a mesma abordagem várias vezes e conta os vereditos; a decisão é a frequência. Tree of Thoughts pede abordagens declaradamente diferentes e as pontua por um critério escrito no prompt; a decisão é a nota.

Isso dá à avaliação um papel que a votação não tem. O critério é um texto sob controle da aplicação e entra somente na chamada de avaliação: trocá-lo repontua as mesmas candidatas, e o exercício 5 mede o efeito dessa troca sobre a seleção.

O avaliador é o mesmo modelo, e a nota é uma saída dele, sujeita aos mesmos erros do raciocínio que avalia. A execução pelo `gemini-3.5-flash-lite` citada na subseção 8.3 é esse caso: a abordagem que ignora a indivisibilidade da aula recebeu a nota máxima do mesmo modelo que, no Chain-of-Thought da seção 3, tinha descartado esse caminho. Uma verificação independente do critério, por uma função ou por um segundo modelo, separa a nota da resposta do modelo que a gerou.

---

# 9. Comparação das técnicas

As sete técnicas produzem raciocínio em formatos diferentes e a um custo diferente em chamadas de modelo. A coluna de chamadas usa `n` para o número de amostras da seção 5 e `k` para o número de subproblemas ou candidatos.

| Técnica | Forma do raciocínio | Chamadas de modelo | Decisão final |
|---|---|---|---|
| Few-Shot | texto livre, no formato dos exemplos | 1 | o que o modelo escrever |
| Chain-of-Thought | campo de texto no esquema | 1 | campo booleano da mesma resposta |
| Self-Ask | lista de pares pergunta e resposta | 1 | campo booleano da mesma resposta |
| Self-Consistency | `n` cadeias completas | `n` | votação entre os vereditos |
| ReAct | histórico de mensagens com observações | 1 por turno | última mensagem do histórico |
| Least-to-Most | subproblemas resolvidos em ordem | 1 + `k` + 1 | chamada de conclusão |
| Tree of Thoughts | candidatos com nota e justificativa | 1 + `k` + 1 | maior nota |

Duas leituras da tabela orientam a escolha. A primeira é que o raciocínio exposto não separa as técnicas em custo: Chain-of-Thought, Self-Ask e ReAct produzem os passos dentro de uma única chamada, e exibi-los custa os tokens que já foram gerados. As chamadas se multiplicam pela repetição, em Self-Consistency, e pela divisão em etapas, em Least-to-Most e Tree of Thoughts.

A segunda é que as técnicas se combinam. A decomposição da seção 7 produz subproblemas que a votação da seção 5 pode resolver um a um, o laço da seção 6 pode encerrar com o esquema da seção 3, e a avaliação da seção 8 pode pontuar as cadeias da seção 5 em vez de contá-las.

Nenhuma coluna da tabela é uma medida de acerto. Nas saídas gravadas neste arquivo, as sete técnicas chegaram à conclusão correta. A conclusão de cada uma não se mantém entre execuções: em medições anteriores do mesmo arquivo, com o mesmo problema, a subseção 7.1 chegou à conclusão oposta em cinco execuções pelo `gpt-oss:120b` e em duas pelo `gemini-3.5-flash-lite`, e a seção 8 em uma execução pelo Gemini. As seções 3, 4 e 6 chegaram à conclusão correta em todas as medições.

As técnicas que variaram são as que decidem o caminho de cálculo em uma chamada dedicada. Nas seções 7 e 8, a saída dessa chamada passa às seguintes sem conferência, e um erro ali não é revisto depois; a mesma pergunta resolvida em uma chamada só, nas seções 3 e 4, manteve a conclusão em todas as medições. O número de chamadas por pergunta não acompanha o acerto: a técnica de mais chamadas por pergunta é a que mais variou.

---

# 10. Exercícios de fixação

Os exercícios modificam exemplos já executados. Use as células vazias abaixo de cada enunciado.

## Exercício 1 — Rótulos fixados pelos exemplos

Reescreva `EXEMPLOS_COM_RACIOCINIO` trocando os rótulos `Raciocínio:` e `Resposta:` por `Passos:` e `Veredito:`, sem alterar as contas dos dois exemplos. Refaça a chamada da subseção 2.1, com a mesma instrução de sistema, e reporte quais rótulos vieram na resposta e em que formato.

## Exercício 2 — Margem da votação

Execute `self_consistency` com `amostras=3` sobre `PROBLEMA`. Escreva uma função que receba o dicionário devolvido e um limite entre 0 e 1, e devolva o veredito quando a razão entre `votos` e o número de cadeias alcançar o limite, e o texto `verificação humana` quando não alcançar. Aplique-a ao resultado com os limites 0.6 e 1.0 e reporte os dois retornos.

## Exercício 3 — Raciocínio do passo no ReAct

Execute `laco_react` passando em `modelo_base` um modelo criado por `criar_modelo(expor_raciocinio=True)`. Aplique `raciocinio_interno` a cada `AIMessage` do histórico devolvido e reporte em qual mensagem o raciocínio anterior ao pedido de chamada apareceu.

## Exercício 4 — Contexto acumulado

Acrescente a `least_to_most` a impressão da string `contexto` antes de cada chamada de resolução, e execute a função sobre `PROBLEMA`. Reporte em qual subproblema a resposta do primeiro aparece pela primeira vez no contexto, e quantas linhas o contexto tem na última chamada.

## Exercício 5 — Critério da avaliação

Refaça as três avaliações da subseção 8.1 sobre a mesma lista `candidatas`, com um critério que peça a nota pelo número de contas que a abordagem exige e que continue pedindo o preenchimento dos campos `nota` e `justificativa`. Guarde as avaliações novas em outra lista, reporte a abordagem que `max` seleciona com elas, e compare as três notas com as de `notas`, da subseção 8.1.

## Exercício 6 — Escolha por custo e formato

Sem executar código, indique qual das sete técnicas atende cada situação abaixo, usando as colunas de chamadas de modelo e de forma do raciocínio da seção 9. Justifique cada escolha em uma frase.

1. Uma resposta cuja conclusão precisa ser lida por um `if` do programa, com uma chamada de modelo e o menor número de tokens gerados.
2. Uma decisão em que o programa precisa medir o quanto o modelo divergiu de si mesmo.
3. Uma pergunta cuja resposta depende de um valor que só uma função Python calcula.
4. Um problema em que caminhos de cálculo diferentes levam a conclusões diferentes, e o critério de escolha entre eles pode ser escrito.
5. Uma resposta em que cada passo do cálculo precisa ser conferido em separado, sem custo de chamadas adicionais.

---

# 11. Resumo da aula

O notebook percorreu sete formas de obter os passos de raciocínio de um modelo, da que só orienta o formato por exemplos até a que gera candidatos e os pontua. Em todas, o raciocínio que a aplicação usa chega dentro da resposta, em um formato declarado pelo código.

| Conceito | Ideia principal |
|---|---|
| Raciocínio interno | Sequência gerada antes da resposta, cobrada em tokens e recuperada por um parâmetro de cada provedor. |
| Few-Shot Prompting | Exemplos completos no prompt fixam o formato e o método da resposta, sem descrever a tarefa. |
| Chain-of-Thought | Passos e conclusão em campos de um esquema, produzidos em uma chamada. |
| Self-Ask | Raciocínio em pares de subpergunta e resposta, cada um verificável em separado. |
| Self-Consistency | Cadeias amostradas de forma independente e decididas por votação, com a margem como medida. |
| ReAct | Raciocínio no histórico de mensagens, com observações vindas de funções Python. |
| Least-to-Most | Decomposição em uma chamada e resolução em ordem de dependência nas seguintes; a lista fixa o caminho de cálculo das etapas posteriores, e é o ponto onde a aplicação intervém. |
| Tree of Thoughts | Candidatos gerados, pontuados por um critério declarado e selecionados pela nota. |
| `method="function_calling"` | Forma de saída estruturada aceita pelos dois provedores do notebook. |
| Retorno vazio do parser | `None` no lugar do objeto, tratado por repetição da chamada. |

## 11.1 Checklist de compreensão

Questões de verificação:

1. O que separa o raciocínio interno de um modelo do raciocínio escrito na resposta, e onde cada provedor entrega o primeiro?
2. Por que uma resposta de uma palavra pode consumir centenas de tokens de saída?
3. O que muda na saída quando os exemplos de um prompt few-shot passam a incluir os passos do cálculo?
4. Que vantagem um campo booleano de veredito tem sobre a leitura da conclusão em texto?
5. Por que a instrução do sistema repete os nomes dos campos já descritos no esquema Pydantic?
6. O que a contagem de votos do Self-Consistency mede, e o que ela não verifica?
7. Por que o raciocínio do ReAct pode conter um valor que o modelo não calculou?
8. Que verificação o Least-to-Most permite entre a decomposição e a resolução, por que um subproblema ausente da lista não é recuperado nas etapas seguintes, e o que a frase acrescentada em 7.3 muda na lista?

## 11.2 Próximos passos

- Verificação de uma cadeia de raciocínio por uma segunda chamada, que confere as contas antes de a conclusão ser aceita;
- subperguntas respondidas por ferramentas, com o resultado da função no lugar da resposta gerada;
- avaliação das técnicas sobre um conjunto de perguntas com resposta conhecida, medindo acerto e número de chamadas;
- limite de contexto quando o histórico de raciocínio passa da janela do modelo;
- busca em mais de um nível na árvore de candidatos, com retorno ao nível anterior quando um ramo é reprovado.

---

# 12. Referências

- Brown et al., Language Models are Few-Shot Learners: https://arxiv.org/abs/2005.14165
- Wei et al., Chain-of-Thought Prompting Elicits Reasoning in Large Language Models: https://arxiv.org/abs/2201.11903
- Press et al., Measuring and Narrowing the Compositionality Gap in Language Models: https://arxiv.org/abs/2210.03350
- Wang et al., Self-Consistency Improves Chain of Thought Reasoning in Language Models: https://arxiv.org/abs/2203.11171
- Yao et al., ReAct: Synergizing Reasoning and Acting in Language Models: https://arxiv.org/abs/2210.03629
- Zhou et al., Least-to-Most Prompting Enables Complex Reasoning in Large Language Models: https://arxiv.org/abs/2205.10625
- Yao et al., Tree of Thoughts: Deliberate Problem Solving with Large Language Models: https://arxiv.org/abs/2305.10601
- LangChain, saída estruturada: https://docs.langchain.com/oss/python/langchain/structured-output
- LangChain, mensagens: https://docs.langchain.com/oss/python/langchain/messages
- Referência de API, `ChatOllama`: https://reference.langchain.com/python/langchain-ollama/chat_models/ChatOllama
- Referência de API, `ChatGoogleGenerativeAI`: https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI
- Gemini, controle de raciocínio: https://ai.google.dev/gemini-api/docs/thinking
- Ollama, documentação: https://docs.ollama.com